# GPT-2 Training on Google Colab (JAX/Flax + TPU v5e-1)

## Setup: Select TPU v5e-1 Runtime
- Go to Runtime → Change runtime type
- Select "TPU v5e-1" under Hardware accelerator
- Click Save

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted!")

## Install Dependencies

In [ ]:
!pip install -q jax[tpu]==0.4.20 flax optax datasets tiktoken
print("Dependencies installed!")

## Setup Project & Train

In [ ]:
import sys
from google.colab import drive

WORK_DIR = '/content/drive/My Drive/mygpt-2-jax'
sys.path.insert(0, WORK_DIR)

import importlib.util

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, f'{WORK_DIR}/{path}')
    m = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(m)
    return m

# Load all modules
config_mod = load_module('config', 'config.py')
train_mod = load_module('train', 'train.py')
main_mod = load_module('main', 'main.py')

GPTConfig = config_mod.GPTConfig
SimpleTokenizer = load_module('tokenizer', 'tokenizer.py').SimpleTokenizer
create_dataset = train_mod.create_dataset
train = train_mod.train

import jax
import os
os.environ['JAX_PLATFORMS'] = 'tpu'

print(f"JAX devices: {jax.devices()}")

# Config
config = GPTConfig(
    d_model=512,
    num_heads=16,
    num_layers=12,
    max_seq_len=512,
    batch_size=16,
    grad_accum_steps=2,
    max_steps=20000,
    warmup_steps=500,
    learning_rate=1e-3,
    weight_decay=0.01,
)

tokenizer = SimpleTokenizer()

# Load data
print("Loading full WikiText-103...")
texts = main_mod.load_training_data()
dataset = create_dataset(texts, tokenizer, config.max_seq_len, config.batch_size)
print(f'Dataset: {len(dataset):,} batches')

# Train
print(f'\nTraining on TPU...\nModel: {config.d_model}d × {config.num_layers} layers × {config.num_heads} heads')
params = train(config, dataset, save_dir='/content/checkpoints')

# Save
import pickle
with open('/content/gpt_model.pkl', 'wb') as f:
    pickle.dump(params, f)
print('Done!')

## Save Checkpoints to Drive

In [ ]:
!cp /content/gpt_model.pkl '/content/drive/My Drive/gpt_model.pkl'
print("Model saved to Drive!")

## Test Model - Interactive Chat

In [ ]:
import pickle
import jax
import jax.numpy as jnp
from gpt import GPT

# Load model params
with open('/content/gpt_model.pkl', 'rb') as f:
    params = pickle.load(f)

model = GPT(config)

# Chat function
def generate(prompt: str, max_tokens: int = 50, temperature: float = 0.7):
    tokens = tokenizer.encode(prompt)
    input_ids = jnp.array([tokens])

    for _ in range(max_tokens):
        if input_ids.shape[1] > config.max_seq_len:
            input_ids = input_ids[:, -config.max_seq_len:]

        logits, _ = model.apply({'params': params}, input_ids, training=False)
        logits = logits[:, -1, :] / temperature

        probs = jax.nn.softmax(logits, axis=-1)
        next_token = jax.random.choice(jax.random.PRNGKey(0), config.vocab_size, p=probs[0])
        input_ids = jnp.concatenate([input_ids, jnp.array([[next_token]])], axis=1)

    return tokenizer.decode(input_ids[0].tolist())

# Test
print(generate("Hello, how are you?"))
print()
print(generate("Once upon a time"))